# OCR, layout, reading order, and visual descriptions

Pause the web app while running GPU notebooks. These cells use the same modules as FastAPI.


Source mapping: L6 local notebook cells 6–8 (OCR), 14–16 (LayoutReader), 22–24 (layout detection), 32 (crops), and 36–42 (VLM prompts/calls). This project runs layout detection before reading order and includes OCR text in every visual prompt. It omits the L6 question-answering agent.


In [ ]:
from pathlib import Path
import sys
# Find the standalone project whether Jupyter starts at its root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app/config.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from app.config import Settings
from app.processors import Processor
from app.files import atomic_text, save_json
settings = Settings(root=ROOT)
settings.initialize()
processor = Processor(settings, progress=print)



In [ ]:
source = ROOT / "input/image/documents/L6/output_7_1.png"
category = "image"  # Use "pdf" or "docs" with your own corresponding input path.
if not source.is_file():
    raise FileNotFoundError("Add an image or choose an existing source first.")



Inspect the stages separately for learning. OCR returns lines, coordinates and confidence. Each stage loads and releases its own model, avoiding simultaneous GPU residency.


In [ ]:
from PIL import Image
if category == "image":
    processor.engines.unload_vision()
    regions = processor.engines.ocr(source)
    layout = processor.engines.layout(source)
    with Image.open(source) as image:
        indexes = processor.engines.order(regions, image.width, image.height)
    print("Layout:", layout)
    print("Reading order:", indexes)
    print("\n".join(regions[i]["text"] for i in indexes))
    
else:
    print("For PDF/DOCX, use the complete adapter below; it renders pages first.")


Now run the complete shared adapter (this repeats the demonstration stages). PDFs render page by page; DOCX renders through LibreOffice. Visual crops and their OCR text go to the VLM. Generated descriptions remain labeled unverified.


In [ ]:
result = processor.process(source, category)
print("Review notes:", result.warnings)
print("Devices:", result.devices)



In [ ]:
# Write one text output with its original extension and relative path preserved.
relative = source.relative_to(ROOT / "input")
output = ROOT / "output" / (relative.as_posix() + ".txt")
atomic_text(output, result.text(relative.as_posix()))
save_json(ROOT / "runtime" / "notebook-result.json", {
    "source": str(relative), "warnings": result.warnings,
    "devices": result.devices, "details": result.details,
})
print(output)
print(output.read_text(encoding="utf-8")[:5000])

